# Train Coco's voice on Kaggle (Pocket TTS)

Fine-tunes Kyutai's Pocket TTS on Coco's clips, following Kyutai's own training code
(`training/README.md` in kyutai-labs/pocket-tts), in two stages:

1. **Teacher**: the 24-layer English model (`english_2026-09_24l`) learns Coco's voice from her clips.
2. **Student**: the fast 6-layer model Coco actually runs (`english_2026-09`, the teacher's own student) is
   distilled from that teacher. It starts from the released 6-layer weights, so this is a short run.

Built to run **headless**: **Save Version > Save & Run All (Commit)**, then close the browser. Each run trains
for up to `TRAIN_HOURS` and leaves in its **Output**:

- `export/coco-pocket.safetensors` + `coco-pocket.yaml` + `coco-reference.wav`: the voice, for the repo's `models/pocket/`
- `listen/`: the same four lines as the voice-copy test, from the student (`student_*.wav`) and from the
  teacher during training (`teacher_*.wav`)
- `runs/`: the training state (the next run continues from it)
- `logs/`: the training logs

**One-time setup**
1. The `coco-voice` dataset from the Piper notebook: **Add Input** > it. Any layout works (the zip, the unpacked
   zip, or the `piper/` folder).
2. **Add-ons > Secrets**: add `HF_TOKEN` (a Hugging Face **Read** token from the account that was granted
   access to `kyutai/pocket-tts`) and switch it on for this notebook. The training needs the gated weights.
3. Settings (right panel): **Accelerator: GPU T4 x2**, **Internet: on**.

**Continuing:** if the log says a stage didn't reach its steps, Add Input > **Your Work** > this notebook's
output (newest version), then Save & Run All again. It picks up where it stopped.

In [ ]:
# ---- Settings ----
TRAIN_HOURS = 8.0       # Training time per run (setup and the final export come on top, ~30 min). Kaggle caps GPU
                        # runs (9-12 h, check your account) and a run that hits the cap loses its whole output.
TEACHER_SHARE = 0.6     # The teacher gets at most this share of TRAIN_HOURS; the student gets the rest
TEACHER_STEPS = 2000    # 4,330 clips at 64 per step is ~68 steps a pass: 2000 is ~30 passes over her clips
STUDENT_STEPS = 2000
PRECISION = "auto"      # "auto": bf16 on newer GPUs, fp16 on T4/P100. If the loss shows nan or the log says
                        # "non-finite gradient", set "fp32" (slower) and run again: it resumes.
POCKET_COMMIT = "bc4d640c925a3782092a841e9bc7ba442a65a838"  # pocket-tts version this notebook was written for
REF_CLIP = "dbrev_line_00000123"  # the voice prompt for the listen files (same as the local voice-copy test)
VALID_CLIPS = 128       # clips held out to measure the loss on

import glob, json, os, re, shutil, subprocess, sys, time
WORK = "/kaggle/working"                        # only this is kept after a run
RUNS, LISTEN, EXPORT, LOGS = (f"{WORK}/{d}" for d in ("runs", "listen", "export", "logs"))
REPO, DATA, PT, SCRATCH = "/content/pocket-tts", "/content/data", "/content/pt", "/content/scratch"
for d in (RUNS, LISTEN, EXPORT, LOGS, DATA, PT, SCRATCH):
    os.makedirs(d, exist_ok=True)

LINES = [  # the voice-copy test's lines, so the two can be compared directly
    "Hey Palmtree. I moved your dentist appointment to Thursday at three, so you're clear tomorrow.",
    "Okay, so the build failed again. Want me to dig through the log, or are we calling it a night?",
    "Honestly? That's a great idea. I'll set a reminder before you forget it.",
    "Found it. Three results, but only the second one looks like it actually answers your question.",
]

def last_step(run_dir):
    steps = [int(m.group(1)) for p in glob.glob(f"{run_dir}/checkpoint_*.pt")
             if (m := re.search(r"checkpoint_(\d+)\.pt$", p))]
    return max(steps, default=0)

def latest_checkpoint(run_dir):
    s = last_step(run_dir)
    return f"{run_dir}/checkpoint_{s:08d}.pt" if s else None

In [ ]:
# ---- 1. GPU, internet, Hugging Face token, pocket-tts (pinned version) ----
!nvidia-smi --query-gpu=name,memory.total --format=csv
import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=15)
except Exception as e:
    raise RuntimeError("No internet: turn it on in Settings (right panel) > Internet") from e
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError("No HF_TOKEN secret: Add-ons > Secrets > add HF_TOKEN, and switch it on for this notebook") from e

!test -d {REPO} || git clone -q https://github.com/kyutai-labs/pocket-tts.git {REPO}
!cd {REPO} && git fetch -q --depth 1 origin {POCKET_COMMIT} 2>/dev/null; cd {REPO} && git checkout -q {POCKET_COMMIT}
# The package plus the training extras (Kyutai's "dev" group, minus docs/test tools). Kaggle's torch stays.
!pip -q install -e {REPO} pyyaml soundfile sphn transformers tqdm jiwer

import torch
NGPU = torch.cuda.device_count()
assert NGPU, "No GPU: Settings (right panel) > Accelerator > GPU T4 x2"
CAP = torch.cuda.get_device_capability(0)
AMP = {"auto": "bfloat16" if CAP[0] >= 8 else "float16", "bf16": "bfloat16", "fp16": "float16", "fp32": "float32"}[PRECISION]
print(f"{NGPU} x {torch.cuda.get_device_name(0)} (compute {CAP[0]}.{CAP[1]}), training in {AMP}")

# Gated weights: fails here, not an hour in, if the token can't read them
from huggingface_hub import hf_hub_download
try:
    for f, rev in [("languages/english_2026-09_24l/model.safetensors", "983151f13aaeab1b13c1e5e3c2c383d49a9edf3f"),
                   ("languages/english_2026-09/model.safetensors", "983151f13aaeab1b13c1e5e3c2c383d49a9edf3f")]:
        hf_hub_download("kyutai/pocket-tts", f, revision=rev)
except Exception as e:
    raise RuntimeError("Can't download kyutai/pocket-tts: is HF_TOKEN from the account that was granted access, "
                       "and is it a Read token (or fine-grained with gated-repo read access)?") from e
print("Gated weights OK")

In [ ]:
# ---- 2. Patches for older GPUs (T4, P100: no bf16) ----
# Kyutai's trainer is written for bf16 GPUs (A100/H100). On older GPUs: autocast in fp16 with a gradient scaler
# (or fp32), and the Muon optimizer's orthogonalization and the aligner in fp32.
def patch(path, edits):
    src = open(path).read()
    for old, new, count in edits:
        if src.count(old) != count:
            raise RuntimeError(f"{path}: expected {count}x {old!r}, found {src.count(old)}. "
                               "The code isn't what this notebook was written for; nothing changed.")
        src = src.replace(old, new)
    open(path, "w").write(src)

TRAIN = f"{REPO}/training/train.py"
if "AMP_DTYPE" not in open(TRAIN).read():
    patch(TRAIN, [
        ("import torch\nfrom torch import nn\n",
         "import torch\nfrom torch import nn\n\n"
         "AMP_DTYPE = getattr(torch, os.environ.get(\"POCKET_AMP_DTYPE\", \"bfloat16\"))\n"
         "SCALER = torch.amp.GradScaler(\"cuda\", enabled=AMP_DTYPE == torch.float16)\n", 1),
        ("dtype=torch.bfloat16, enabled=device.type == \"cuda\"",
         "dtype=AMP_DTYPE, enabled=device.type == \"cuda\" and AMP_DTYPE != torch.float32", 2),
        ("(loss / args.grad_accum_steps).backward()",
         "SCALER.scale(loss / args.grad_accum_steps).backward()", 1),
        ("        grad_norm = torch.nn.utils.clip_grad_norm_(",
         "        SCALER.unscale_(optimizer)\n        grad_norm = torch.nn.utils.clip_grad_norm_(", 1),
        # fp16: a step with inf/nan gradients is skipped and the scale lowered (normal now and then)
        ("if not torch.isfinite(grad_norm):", "if not torch.isfinite(grad_norm) and not SCALER.is_enabled():", 1),
        ("        optimizer.step()\n        if ema is not None:",
         "        SCALER.step(optimizer)\n        SCALER.update()\n        if ema is not None:", 1),
    ])
if CAP[0] < 8:
    MUON = f"{REPO}/training/modules/muon.py"
    if "G.bfloat16()" in open(MUON).read():
        patch(MUON, [("X = G.bfloat16()", "X = G.float()", 1),
                     ("dtype=torch.bfloat16, device=items[0][2].device", "dtype=torch.float32, device=items[0][2].device", 1)])
    ALIGN = f"{REPO}/training/scripts/align_data.py"
    if "use_bf16 = device.type == \"cuda\"\n" in open(ALIGN).read():
        patch(ALIGN, [("use_bf16 = device.type == \"cuda\"\n", "use_bf16 = False\n", 1)])
!cd {REPO} && python -m py_compile training/train.py training/modules/muon.py training/scripts/align_data.py && echo "Patched OK"

In [ ]:
# ---- 3. Coco's clips -> manifests -> word timings (forced alignment, as Kyutai's README describes) ----
import random, soundfile as sf
# Any layout works: the wavs are found by file name wherever they are (read in place, no copying), and the
# metadata.csv that names the most of them is used. Its lines are "<id>|<text>" or "<id>.wav|<text>".
def find_wavs(root):
    return {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(f"{root}/**/*.wav", recursive=True)}
def clip_id(line):
    name = line.split("|", 1)[0].strip()
    return os.path.splitext(os.path.basename(name))[0] if name.lower().endswith(".wav") else name

WAVS = find_wavs("/kaggle/input")
if not WAVS:   # only the zip was uploaded (Kaggle didn't unpack it)
    zips = sorted(glob.glob("/kaggle/input/**/coco-voice-dataset*.zip", recursive=True))
    if not zips:
        raise FileNotFoundError("No dataset: Add Input > the coco-voice dataset (with the clips and metadata.csv)")
    subprocess.run(["unzip", "-q", "-o", zips[0], "-d", DATA], check=True)
    WAVS = find_wavs(DATA)
metas = glob.glob("/kaggle/input/**/metadata.csv", recursive=True) + glob.glob(f"{DATA}/**/metadata.csv", recursive=True)
if not metas:
    raise FileNotFoundError("No metadata.csv in the attached dataset")
def matches(path):
    return sum(clip_id(l) in WAVS for l in open(path, encoding="utf-8") if "|" in l)
META = max(metas, key=matches)
print(f"{len(WAVS)} wavs found; using {META} ({matches(META)} of its lines have a wav)")

rows, missing = [], 0
for line in open(META, encoding="utf-8"):
    parts = line.rstrip("\r\n").split("|")
    if len(parts) < 2 or not parts[1].strip():
        continue
    wav = WAVS.get(clip_id(line))
    if wav is None:
        missing += 1   # the full metadata.csv also lists the clips the dataset step rejected
        continue
    text = parts[1].strip().replace("’", "'").replace("‘", "'")
    rows.append({"path": wav, "duration": round(sf.info(wav).duration, 3), "transcript": text})
if len(rows) < 1000:
    raise RuntimeError(f"Only {len(rows)} clips have both a wav and a line in {META}: check the dataset")
REF_WAV = WAVS[REF_CLIP]
random.Random(0).shuffle(rows)
valid = [r for r in rows if REF_CLIP not in r["path"]][:VALID_CLIPS]
train = [r for r in rows if r not in valid]
for name, part in (("train", train), ("valid", valid)):
    with open(f"{PT}/{name}.jsonl", "w") as f:
        f.writelines(json.dumps(r) + "\n" for r in part)
print(f"{len(train)} training clips ({sum(r['duration'] for r in train) / 3600:.1f} h), {len(valid)} held out, "
      f"{missing} listed without a wav (skipped)")

for name in ("train", "valid"):
    out = f"{PT}/{name}_aligned.jsonl"
    if not os.path.exists(out):
        subprocess.run([sys.executable, "-m", "training.scripts.align_data", f"{PT}/{name}.jsonl", out + ".part",
                        "--device", "cuda"], cwd=REPO, check=True)
        os.replace(out + ".part", out)
    n = sum(1 for _ in open(out))
    print(f"{name}: {n} clips aligned")
    assert n > 0.9 * len(train if name == "train" else valid), "Too many clips failed to align: see the output above"

In [ ]:
# ---- 4. Continue from an earlier run, if its output is attached ----
def runs_in(root):
    return [os.path.dirname(p) for p in glob.glob(f"{root}/**/runs/teacher/args.yaml", recursive=True)]
attached = runs_in("/kaggle/input")
if attached and not last_step(f"{RUNS}/teacher"):
    best = max(attached, key=lambda r: (last_step(f"{r}/teacher"), last_step(f"{r}/student")))
    print(f"Copying the earlier run from {best} ...")
    shutil.copytree(best, RUNS, dirs_exist_ok=True)
print(f"Teacher: step {last_step(f'{RUNS}/teacher')} of {TEACHER_STEPS}")
print(f"Student: step {last_step(f'{RUNS}/student')} of {STUDENT_STEPS}")
if not attached and not last_step(f"{RUNS}/teacher"):
    print("No earlier run attached: starting fresh. (To continue one, Add Input > Your Work > this notebook's output.)")

In [ ]:
# ---- 5. Train: the teacher, then the student ----
import signal, yaml
GPU_BATCH = {"teacher": 16, "student": 32}   # per GPU; lower these if you get "CUDA out of memory"
SAMPLES = LINES[:3]

def common(name, steps):
    batch = GPU_BATCH[name]
    return {
        "data": {"train_jsonl": f"{PT}/train_aligned.jsonl", "valid_jsonl": f"{PT}/valid_aligned.jsonl",
                 "max_duration_sec": 30.0, "max_voice_prompt_sec": 5.0, "loader_procs": 2},
        "flow": {"type": "lsd", "kwargs": {}},
        "run_dir": f"{RUNS}/{name}",
        "batch_size": batch,
        "grad_accum_steps": max(1, 64 // (batch * NGPU)),   # 64 rows per step, as Kyutai's configs
        "max_steps": steps,
        "compile": False,          # torch.compile warm-up costs more than it saves on runs this short
        "log_freq": 25, "valid_freq": 250, "num_valid_batches": 8,
        "ckpt_freq": 250, "num_ckpt_keep": 1,
        "sample_sentences": SAMPLES, "sample_freq": 500,
    }

# Kyutai's finetune.yaml, for one voice: sampler head frozen (as their own robustness fine-tune of this teacher
# did), so the 6-layer student's head still fits; latent statistics left as released.
TEACHER = common("teacher", TEACHER_STEPS) | {
    "model_config": "pocket_tts/config/english_2026-09_24l.yaml",
    "start_from_pretrained": True, "freeze_head": True,
    "flow_batch_multiplier": 4, "eos_loss_weight": 0.1, "text_dropout": 0.2, "voice_dropout": 0.2,
    "optim": {"lr": 2e-5, "muon_lr": 5e-4, "schedule": "constant", "warmup_steps": 200},
    "stats_ema_steps": 0, "ema_decay": 0.999, "sample_cfg_coef": 2.0,
}
# Kyutai's depth_distill.yaml, shortened: the student starts from the released 6-layer weights, not from scratch.
STUDENT = common("student", STUDENT_STEPS) | {
    "model_config": "pocket_tts/config/english_2026-09.yaml",
    "start_from_pretrained": False,
    "distill_cfg_coef": 2.0,
    "distill_teacher_config": "pocket_tts/config/english_2026-09_24l.yaml",
    "student_init_from": "hf://kyutai/pocket-tts/languages/english_2026-09/model.safetensors@983151f13aaeab1b13c1e5e3c2c383d49a9edf3f",
    "text_dropout": 0.0, "voice_dropout": 0.0,
    "optim": {"lr": 1e-4, "muon_lr": 1e-3, "schedule": "cosine", "warmup_steps": 200},
    "ema_decay": 0.999, "sample_cfg_coef": 1.0,
}

def log_tail(path, n=1, match="step "):
    try:
        lines = [l for l in open(path, errors="replace").read().splitlines() if match in l]
        return lines[-n:]
    except OSError:
        return []

def train(name, cfg, deadline):
    run_dir = cfg["run_dir"]
    if last_step(run_dir) >= cfg["max_steps"]:
        print(f"{name}: already at step {last_step(run_dir)}, done")
        return
    if time.time() >= deadline:
        print(f"{name}: no time left in this run (step {last_step(run_dir)} of {cfg['max_steps']})")
        return
    config = f"{SCRATCH}/{name}.yaml"
    open(config, "w").write(yaml.safe_dump(cfg, sort_keys=False))
    log_path = f"{LOGS}/{name}.log"
    print(f"{name}: training from step {last_step(run_dir)} to {cfg['max_steps']}, "
          f"until {time.strftime('%H:%M', time.localtime(deadline))} at the latest. Log: logs/{name}.log", flush=True)
    log = open(log_path, "a")
    proc = subprocess.Popen(["torchrun", "--standalone", f"--nproc-per-node={NGPU}", "-m", "training.train", config],
                            cwd=REPO, stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
                            env=os.environ | {"POCKET_AMP_DTYPE": AMP, "PYTHONUNBUFFERED": "1"})
    t0, last_report, stop_at = time.time(), 0, None
    while proc.poll() is None:
        time.sleep(20)
        now = time.time()
        if now - last_report > 900:   # every 15 minutes
            last_report = now
            print(f"  [{(now - t0) / 3600:.2f} h] " + " ".join(log_tail(log_path)) [-220:], flush=True)
        if now >= deadline and stop_at is None:
            stop_at = now   # the trainer checkpoints after its current step and exits
            open(f"{run_dir}/STOP", "w").close()
            print(f"  Time's up for {name}: saving a checkpoint and stopping...", flush=True)
        if stop_at and now - stop_at > 900:
            print("  It didn't stop in 15 minutes: killing it (the previous checkpoint is kept).")
            os.killpg(proc.pid, signal.SIGKILL)
            proc.wait()
    log.close()
    print(f"{name}: stopped at step {last_step(run_dir)} of {cfg['max_steps']} after {(time.time() - t0) / 3600:.2f} h, "
          f"exit code {proc.returncode}")
    for line in log_tail(log_path, 3, "valid @"):
        print("  " + line[-200:])
    if proc.returncode != 0 and stop_at is None:
        print("".join(open(log_path, errors="replace").readlines()[-40:]))
        raise RuntimeError(f"{name} training failed: the end of logs/{name}.log is above")

T0 = time.time()
train("teacher", TEACHER, T0 + TRAIN_HOURS * 3600 * TEACHER_SHARE)
teacher_ckpt = latest_checkpoint(f"{RUNS}/teacher")
assert teacher_ckpt, "The teacher saved no checkpoint: see logs/teacher.log"
teacher_step = last_step(f"{RUNS}/teacher")

# A student distilled from an older teacher starts over
tag = f"{RUNS}/student/TEACHER_STEP"
if os.path.exists(tag) and open(tag).read().strip() != str(teacher_step):
    print(f"The teacher moved on (step {open(tag).read().strip()} -> {teacher_step}): the student starts over")
    shutil.rmtree(f"{RUNS}/student")
os.makedirs(f"{RUNS}/student", exist_ok=True)
open(tag, "w").write(str(teacher_step))
STUDENT["distill_teacher_weights"] = teacher_ckpt
train("student", STUDENT, T0 + TRAIN_HOURS * 3600)

In [ ]:
# ---- 6. Export the voice and write the listen files ----
student_step = last_step(f"{RUNS}/student")
assert os.path.exists(f"{RUNS}/student/model.safetensors"), "The student saved nothing yet: see logs/student.log"
for old in glob.glob(f"{LISTEN}/*") + glob.glob(f"{EXPORT}/*"):
    os.remove(old)
shutil.copy(f"{RUNS}/student/model.safetensors", f"{EXPORT}/coco-pocket.safetensors")   # EMA weights, Mimi included
shutil.copy(REF_WAV, f"{EXPORT}/coco-reference.wav")
cfg = open(f"{REPO}/pocket_tts/config/english_2026-09.yaml").read()
cfg = re.sub(r"(?m)^weights_path_without_voice_cloning:.*\n", "", cfg)
cfg = re.sub(r"(?m)^weights_path:.*$", "weights_path: coco-pocket.safetensors   # next to this file", cfg)
cfg = (f"# Coco's voice: english_2026-09 distilled from a 24-layer teacher fine-tuned on her clips\n"
       f"# (teacher step {teacher_step}, student step {student_step}). Generate at cfg 1, with coco-reference.wav as the voice.\n" + cfg)
open(f"{EXPORT}/coco-pocket.yaml", "w").write(cfg)

# The loader reads weights_path relative to the working directory: give it an absolute one here
open(f"{SCRATCH}/listen.yaml", "w").write(cfg.replace("weights_path: coco-pocket.safetensors",
                                                         f"weights_path: {EXPORT}/coco-pocket.safetensors"))
LISTEN_PY = f"""
import json, sys, time, soundfile as sf
from pocket_tts import TTSModel
lines, out = json.loads(sys.argv[1]), sys.argv[2]
model = TTSModel.load_model(config="{SCRATCH}/listen.yaml")
state = model.get_state_for_audio_prompt("{EXPORT}/coco-reference.wav")
for i, line in enumerate(lines):
    audio = model.generate_audio(state, line).numpy()
    sf.write(f"{{out}}_{{i}}.wav", audio, model.sample_rate)
    print(f"{{out}}_{{i}}.wav: {{len(audio) / model.sample_rate:.1f}} s")
"""
open(f"{SCRATCH}/listen.py", "w").write(LISTEN_PY)
subprocess.run([sys.executable, f"{SCRATCH}/listen.py", json.dumps(LINES), f"{LISTEN}/student_step{student_step}"],
               cwd=REPO, check=True)

# The teacher's own samples (24 layers, guidance 2), written during training
samples = sorted(glob.glob(f"{RUNS}/teacher/samples/step*_*.wav"))
if samples:
    newest = os.path.basename(samples[-1]).split("_")[0]
    for p in glob.glob(f"{RUNS}/teacher/samples/{newest}_*.wav"):
        shutil.copy(p, f"{LISTEN}/teacher_{os.path.basename(p)}")
shutil.copy(REF_WAV, f"{LISTEN}/_REFERENCE.wav")
!du -sh {RUNS} {EXPORT}; ls -la {EXPORT} {LISTEN}

In [ ]:
# ---- 7. Listen here (only useful when running interactively; a committed run saves the files above) ----
from IPython.display import Audio, display
for path in sorted(glob.glob(f"{LISTEN}/*.wav")):
    print(os.path.basename(path)); display(Audio(path))